In [ ]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
N_PEERS = 130
N_ITERS = 10
T_EVENT = 70
REACHABILITY_TARGET = 0.99

TAGS = [f'{t_m}-300' for t_m in [0, 200, 400, 600]]
TAGS.extend(
    [f'300-{t_u}' for t_u in [100, 300, 500, 700]]
)

# reach
DIR_PATHS = [f'../ablation/dev-v2/t_min_{t_m}_unit_300/{N_PEERS}/' for t_m in [0, 200, 400, 600]]
DIR_PATHS.extend(
    [f'../ablation/dev-v2/t_min_300_unit_{t_u}/{N_PEERS}/' for t_u in [100, 300, 500, 700]]
)

# netuse
FILE_PATHS = [f'../ablation/dev-v2/t_min_{t_m}_unit_300/{N_PEERS}/n_sum.txt' for t_m in [0, 200, 400, 600]]
FILE_PATHS.extend(
    [f'../ablation/dev-v2/t_min_300_unit_{t_u}/{N_PEERS}/n_sum.txt' for t_u in [100, 300, 500, 700]]
)

In [ ]:
def parse_reach():
    combined = []
    for dirpath in DIR_PATHS:
        result = []
        for i in range(N_ITERS):
            filepath = dirpath + f'r_{i}.txt'
            with open(filepath, 'r') as f:
                time_init = 0
                for line in f:
                    parts = line.strip().split()
                    if parts[1] == 'N/A':
                        continue
                    
                    time = int(parts[0])
                    if time_init == 0:
                        time_init = time

                    if time - time_init < 71_000: # 71 seconds.
                        continue
                        
                    reachability = float(parts[1])
                    if reachability >= REACHABILITY_TARGET:
                        result.append((time - time_init)/1000 - T_EVENT)
                        break
                
        combined.append(result)
    return combined

In [ ]:
def parse_netuse() -> list[list[float]]:
    # list of (total network usage (MB) of each host in a FILE_PATH)
    combined = []
    for filepath in FILE_PATHS:
        result = []
        with open(filepath, 'r') as f:
            for line in f:
                result.append(float(line.strip())/1000)
        print(f'sample value: {result[0]}')
        combined.append(result)
    return combined

In [ ]:
def draw_plots(x_labels: list[str], dot_data: list[list[float]], box_data: list[list[float]],
               left_ylabel: str = '', right_ylabel: str = ''):
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 6,
        "axes.titlesize": 6,
        "axes.labelsize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 6,
        "figure.titlesize": 6,
    })

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(3.9, 2))
 
    for i, values in enumerate(dot_data):
        jitter = np.random.uniform(-0.2, 0.2, len(values))
        ax1.plot(i + jitter, values, '.', color='black', markersize=2, alpha = 0.1, markerfacecolor='black', markeredgewidth=0)
 
    ax1.set_xticks(range(len(x_labels)))
    ax1.set_xticklabels(x_labels, rotation=60, ha='right')
    ax1.set_ylabel(left_ylabel, labelpad=1)
 
    bp = ax2.boxplot(box_data, labels=x_labels, patch_artist=True,
                flierprops=dict(marker='.', markersize=3, markerfacecolor='black', markeredgewidth=0),
                medianprops=dict(color='black'))
    for patch in bp['boxes']:
        patch.set_facecolor((0, 0, 0, 0.5))
    ax2.set_xticklabels(x_labels, rotation=60, ha='right')
    ax2.set_ylabel(right_ylabel, labelpad=1)

    plt.tight_layout()
    plt.savefig("ablation.jpg", dpi=600, bbox_inches="tight", pad_inches=0)

In [ ]:
netuse_data = parse_netuse()
reach_data = parse_reach()
x_tags = [str(v) for v in TAGS]
draw_plots(x_tags, netuse_data, reach_data, 'Network (MB)', 'Discovery Time (s)')